In [1]:
# %% Imports and setup
import sys, os, random, json, time, traceback
from pathlib import Path
sys.path.append(str(Path('../../src').resolve()))

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (LSTM, Dense, Dropout, Bidirectional,
                                     Conv1D, Flatten, Input, Attention, GlobalAveragePooling1D)
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import MinMaxScaler

from utils.lstm_configs import LSTM_CONFIGS
from utils.variants import VARIANT_DEFS, apply_variant
from utils.eval_metrics import evaluate_predictions

# Reproducibility
SEED = 1
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# %% Paths
BEST_IN = Path("../../data/out/best_models_lstm/in/best_models_lstm.xlsx")
BEST_OUT = Path("../../data/out/best_models_lstm/validation_lstm")
MODELS_DIR = BEST_OUT / "models"
PLOTS_DIR = BEST_OUT / "plots"
PRED_DIR  = BEST_OUT / "predictions"
for d in [BEST_OUT, MODELS_DIR, PLOTS_DIR, PRED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

BASE_DATA_PATH = Path('../../data/out/dataset_final.csv')
VALIDATION_PATH = Path('../../data/out/dataset_validation_final.csv')

TARGET_COLUMN = "PRECIO"
categorical_numeric = ["YEAR","MONTH","DAY","HORA","NIVEL_ENSO","DIA_SEMANA","FESTIVO"]
TIMESTEPS = 24

In [2]:
# %% Helpers
def get_lstm_config(name):
    for c in LSTM_CONFIGS:
        if c["name"] == name:
            return c.copy()
    raise ValueError(f"Config {name} no encontrada en LSTM_CONFIGS")

def prepare_variant_data_for_lstm(df, scaler=None, fit=False, target_column='PRECIO',
                                  timesteps=24,
                                  categorical_numeric=["YEAR","MONTH","DAY","HORA","NIVEL_ENSO","DIA_SEMANA","FESTIVO"]):
    """
    Prepara secuencias para LSTM a partir de un DataFrame ya transformado por la variante.
    - Si fit=True, ajusta el scaler con los datos numéricos.
    - Si fit=False, solo transforma con el scaler ya entrenado.
    Devuelve X_seq, y_seq completos (sin split).
    """
    df = df.drop(columns=['FECHA_HORA'], errors='ignore')

    # Separar numéricas y categóricas
    numeric_cols = [c for c in df.columns if c not in categorical_numeric + [target_column]]
    X_num = df[numeric_cols].values.astype('float32') if numeric_cols else None
    X_cat = df[[c for c in categorical_numeric if c in df.columns]].values.astype('float32')
    y = df[target_column].values.astype('float32')

    # Escalar solo numéricas
    if X_num is not None:
        if fit:
            scaler.fit(X_num)
        X_num_scaled = scaler.transform(X_num)
        X_all = np.concatenate([X_num_scaled, X_cat], axis=1)
    else:
        X_all = X_cat

    # Construir secuencias
    X_seq, y_seq = [], []
    for i in range(len(X_all) - timesteps):
        X_seq.append(X_all[i:i+timesteps, :])
        y_seq.append(y[i+timesteps])
    return np.array(X_seq), np.array(y_seq)

def build_lstm(input_shape, config):
    """Construye un modelo LSTM según la configuración."""
    if "conv1d" in config:
        model = Sequential()
        conv = config["conv1d"]
        model.add(Conv1D(filters=conv["filters"],
                         kernel_size=conv["kernel_size"],
                         activation=conv["activation"],
                         input_shape=input_shape))
        model.add(Flatten())
        for units in config["lstm_layers"]:
            model.add(Dense(units, activation="relu"))
        if config.get("dropout", 0) > 0:
            model.add(Dropout(config["dropout"]))
        model.add(Dense(1))
    elif config.get("attention", False):
        inputs = Input(shape=input_shape)
        x = LSTM(config["layers"][0], return_sequences=True)(inputs)
        context = Attention()([x, x])
        context = GlobalAveragePooling1D()(context)
        if config.get("dropout", 0) > 0:
            context = Dropout(config["dropout"])(context)
        outputs = Dense(1)(context)
        model = Model(inputs, outputs)
    else:
        model = Sequential()
        for i, units in enumerate(config["layers"]):
            return_sequences = i < len(config["layers"]) - 1
            layer = LSTM(units,
                         return_sequences=return_sequences,
                         recurrent_dropout=config.get("recurrent_dropout", 0.0),
                         input_shape=input_shape)
            if config.get("bidirectional", False):
                model.add(Bidirectional(layer))
            else:
                model.add(layer)
            if config.get("dropout", 0) > 0:
                model.add(Dropout(config["dropout"]))
        model.add(Dense(1))
    model.compile(optimizer=config["optimizer"], loss="mse")
    return model

In [3]:
# %% Load best configs and datasets
best_df = pd.read_excel(BEST_IN)

rows = []

scaler = MinMaxScaler()
df_train = pd.read_csv(BASE_DATA_PATH)
df_val = pd.read_csv(VALIDATION_PATH)

df_train["FECHA_HORA"] = pd.to_datetime(df_train["FECHA_HORA"], dayfirst=True, errors="coerce")
df_val["FECHA_HORA"]   = pd.to_datetime(df_val["FECHA_HORA"], dayfirst=True, errors="coerce")

# %% Loop over best configs
for _, row in best_df.iterrows():
    model_name = row["variant"]      # siempre "LSTM"
    config_name = row["model"]       # ej. "cnn_lstm"
    variant = row["config"]          # ej. "v1_original"

    try:
        # Reconstruir datasets por variante
        vdef = next(v for v in VARIANT_DEFS if v["name"] == variant)
        df_train_var, _ = apply_variant(df_train.copy(), vdef, date_col="FECHA_HORA")
        df_val_var, _ = apply_variant(df_val.copy(), vdef, date_col="FECHA_HORA")

        # Config del modelo
        config = get_lstm_config(config_name)

        # Secuencias de entrenamiento (fit scaler) y validación (transform con el mismo scaler)
        X_full, y_full = prepare_variant_data_for_lstm(df_train_var, scaler=scaler, fit=True, timesteps=TIMESTEPS)
        X_val_seq, y_val_seq = prepare_variant_data_for_lstm(df_val_var, scaler=scaler, fit=False, timesteps=TIMESTEPS)

        # Construir y entrenar
        model = build_lstm((X_full.shape[1], X_full.shape[2]), config)
        start = time.time()
        es = EarlyStopping(monitor="loss", patience=5, restore_best_weights=True)
        history = model.fit(
            X_full, y_full,
            epochs=config["epochs"],
            batch_size=config["batch_size"],
            callbacks=[es],
            verbose=0
        )
        elapsed = time.time() - start

        # Predecir y evaluar
        y_pred = model.predict(X_val_seq, verbose=0).ravel()
        metrics = evaluate_predictions(y_val_seq, y_pred)
        metrics.update({
            "variant": variant,
            "config": config_name,
            "train_time_s": elapsed,
            "n_train": len(y_full),
            "n_val": len(y_val_seq),
            "status": "trained"
        })
        rows.append(metrics)

        # Guardar modelo y history
        model.save(MODELS_DIR / f"{variant}_{config_name}.h5")
        with open(MODELS_DIR / f"{variant}_{config_name}_history.json", "w") as f:
            json.dump(history.history, f)

        # Guardar predicciones
        df_preds = pd.DataFrame({"y_real": y_val_seq, "y_pred": y_pred})
        df_preds.to_csv(PRED_DIR / f"predictions_{variant}_{config_name}.csv", index=False)

        # Gráficas
        plt.figure(figsize=(12,5))
        plt.plot(y_val_seq[:200], label="Real")
        plt.plot(y_pred[:200], label="Predicted")
        plt.legend(); plt.tight_layout()
        plt.savefig(PLOTS_DIR / f"pred_{variant}_{config_name}.png"); plt.close()

        plt.figure(figsize=(5,5))
        plt.scatter(y_val_seq, y_pred, s=6, alpha=0.5)
        lims = [min(plt.xlim()[0], plt.ylim()[0]), max(plt.xlim()[1], plt.ylim()[1])]
        plt.plot(lims, lims, 'r--', linewidth=2)
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / f"scatter_{variant}_{config_name}.png"); plt.close()

        resid = y_val_seq - y_pred
        plt.figure(figsize=(12,4))
        plt.plot(resid[:200], color="tab:red")
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / f"resid_{variant}_{config_name}.png"); plt.close()

        print(f"OK {variant}-{config_name}: MAE={metrics.get('MAE'):.2f}")

    except Exception as e:
        tb = traceback.format_exc()
        print(f"ERROR {variant}-{config_name}: {e}")
        traceback.print_exc()
        rows.append({
            "variant": variant,
            "config": config_name,
            "status": "error",
            "error": str(e),
            "traceback": tb
        })

# %% Save consolidated metrics
if rows:
    df_out = pd.DataFrame(rows)
    df_out.to_excel(BEST_OUT / "validation_lstm_metrics.xlsx", index=False)

c:\Users\Camilo\anaconda3\envs\env_tf\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


OK v1_original-cnn_lstm: MAE=58.02


c:\Users\Camilo\anaconda3\envs\env_tf\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


OK v1_original_lags-cnn_lstm: MAE=105.00


c:\Users\Camilo\anaconda3\envs\env_tf\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


...

OK v9_date_range_all_lags-cnn_lstm: MAE=111.84


c:\Users\Camilo\anaconda3\envs\env_tf\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


OK v10_date_range_no_solar-lstm_simple: MAE=17.55


c:\Users\Camilo\anaconda3\envs\env_tf\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


OK v10_date_range_no_solar_lags-cnn_lstm: MAE=122.74
